In [1]:
!rm -rf /root/.cache/huggingface

!pip install -q transformers datasets evaluate scikit-learn pandas numpy matplotlib seaborn
!pip install -q accelerate
!pip install -q sentencepiece
!pip install -q huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00


In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

In [3]:
!pip install -q seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [5]:
import re
import os
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    classification_report
)
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    pipeline
)

# ============================
# 1. Load Datasets
# ============================
TRAIN_CSV = '/content/Gemini 3.5-Flash_Extended_Synthetic_Dataset.csv'
TEST_CSV  = '/content/Test.csv'

df_train = pd.read_csv(TRAIN_CSV)
df_test  = pd.read_csv(TEST_CSV)

print(f"Training samples: {len(df_train)}")
print(f"Test samples:     {len(df_test)}")

# ============================
# 2. Parse BIO Tags (if present)
# ============================
def parse_bio_tags(bio_text):
    tokens, labels = [], []
    for part in bio_text.split():
        match = re.match(r'^(.*?)(?:[,\s।]*)-(B|I)$', part)
        if match:
            token, tag = match.group(1), match.group(2)
            labels.append('B-Product' if tag == 'B' else 'I-Product')
        else:
            token = part
            labels.append('O')
        tokens.append(token)
    return tokens, labels

if 'bio_tagged_review' in df_test.columns:
    df_test['tokens']   = df_test['bio_tagged_review'].apply(lambda x: parse_bio_tags(x)[0])
    df_test['ner_tags'] = df_test['bio_tagged_review'].apply(lambda x: parse_bio_tags(x)[1])
else:
    df_test['tokens'] = df_test['review'].apply(lambda x: x.split())

# ============================
# 3. Zero‑Shot NER (pre‑trained mBERT Bengali NER)
# ============================
NER_MODEL = "sagorsarker/mbert-bengali-ner"

def _safe_from_pretrained(factory, identifier, **kwargs):
    local_dir = os.environ.get('LOCAL_MODEL_DIR', os.path.join(os.getcwd(), 'local_model'))
    try:
        return factory(identifier, **kwargs)
    except Exception as remote_err:
        import warnings
        warnings.warn(f"Remote load failed ({remote_err}); trying local: {local_dir}")
        if os.path.isdir(local_dir):
            return factory(local_dir, local_files_only=True, **kwargs)
        raise RuntimeError(f"Could not load {identifier}.") from remote_err

tokenizer_ner = _safe_from_pretrained(AutoTokenizer.from_pretrained, NER_MODEL)
model_ner = _safe_from_pretrained(
    AutoModelForTokenClassification.from_pretrained,
    NER_MODEL
)
model_id2label = model_ner.config.id2label

# ============================
# 4. Zero‑Shot Sentiment Pipeline (multilingual NLI)
# ============================
sentiment_pipe = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7",
    device=0 if torch.cuda.is_available() else -1
)
candidate_labels = ["negative", "neutral", "positive"]

# ============================
# 5. Helper: get NER predictions per token
# ============================
def get_ner_predictions(df, tokenizer, model):
    all_pred_labels = []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    for _, row in df.iterrows():
        tokens = row['tokens']
        encoding = tokenizer(
            tokens,
            truncation=True,
            is_split_into_words=True,
            return_tensors="pt",
            padding=False
        )
        input_ids = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)
        word_ids = encoding.word_ids(batch_index=0)

        with torch.no_grad():
            logits = model(input_ids, attention_mask=attention_mask).logits
            preds = torch.argmax(logits, dim=2).squeeze().cpu().numpy()

        # Align first sub‑token per word
        token_preds = []
        prev_word_idx = None
        for i, word_idx in enumerate(word_ids):
            if word_idx is None:
                continue
            if word_idx != prev_word_idx:
                token_preds.append(preds[i])
                prev_word_idx = word_idx

        # Convert to label strings
        pred_labels = [model_id2label[lid] for lid in token_preds]
        all_pred_labels.append(pred_labels)
    return all_pred_labels

# ============================
# 6. Run NER evaluation (if ground truth exists)
# ============================
if 'ner_tags' in df_test.columns:
    print("\nComputing NER predictions (zero‑shot)...")
    pred_ner_labels = get_ner_predictions(df_test, tokenizer_ner, model_ner)
    true_ner_labels = df_test['ner_tags'].tolist()

    # Binary: O vs non‑O
    def to_binary(labels):
        return [1 if l != 'O' else 0 for l in labels]

    true_binary = [to_binary(seq) for seq in true_ner_labels]
    pred_binary = [to_binary(seq) for seq in pred_ner_labels]

    flat_true = [item for sublist in true_binary for item in sublist]
    flat_pred = [item for sublist in pred_binary for item in sublist]

    ner_metrics = {
        'precision': precision_score(flat_true, flat_pred, zero_division=0),
        'recall': recall_score(flat_true, flat_pred, zero_division=0),
        'f1': f1_score(flat_true, flat_pred, zero_division=0),
        'accuracy': accuracy_score(flat_true, flat_pred)
    }
    print("\n=== NER (binary O vs non‑O) Test Results ===")
    print(ner_metrics)
else:
    print("\nNo NER ground truth – skipping NER evaluation.")

# ============================
# 7. Sentiment predictions & evaluation (robust label handling)
# ============================
print("\nComputing sentiment predictions (zero‑shot)...")
sentiment_preds = []
for review in df_test['review']:
    result = sentiment_pipe(review, candidate_labels)
    sentiment_preds.append(result['labels'][0].lower())   # force lowercase

if 'sentiment' in df_test.columns:
    # Convert ground truth to lowercase strings
    sent_col = df_test['sentiment']

    # If it's numeric, map to strings
    if pd.api.types.is_numeric_dtype(sent_col):
        # Assume 0=Negative, 1=Neutral, 2=Positive (or whatever mapping)
        # Adjust mapping based on your data
        # We'll try a safe mapping: 0->negative, 1->neutral, 2->positive
        # If you have other values, adjust accordingly.
        label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
        true_sent = sent_col.map(label_map).tolist()
    else:
        # Already string – lowercase and strip
        true_sent = sent_col.astype(str).str.lower().str.strip().tolist()

    # Ensure we have only the three labels; replace any unknown with 'neutral'
    valid_labels = set(candidate_labels)
    true_sent = [l if l in valid_labels else 'neutral' for l in true_sent]

    # Compute metrics
    from sklearn.metrics import accuracy_score, f1_score, classification_report
    acc = accuracy_score(true_sent, sentiment_preds)
    f1 = f1_score(true_sent, sentiment_preds, average='weighted')
    print("\n=== Sentiment Test Results ===")
    print(f"Accuracy: {acc:.4f}")
    print(f"Weighted F1: {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(true_sent, sentiment_preds, zero_division=0))
else:
    print("\nNo sentiment ground truth – skipping sentiment evaluation.")

# ============================
# 8. Save predictions
# ============================
df_results = df_test[['id', 'review']].copy()
if 'ner_tags' in df_test.columns:
    df_results['true_ner'] = df_test['ner_tags'].apply(lambda x: ' '.join(x))
    df_results['pred_ner'] = [' '.join(seq) for seq in pred_ner_labels]
if 'sentiment' in df_test.columns:
    df_results['true_sentiment'] = true_sent if 'true_sent' in locals() else None
df_results['pred_sentiment'] = sentiment_preds
df_results.to_csv("zero_shot_predictions.csv", index=False)
print("\nPredictions saved to zero_shot_predictions.csv")

# ============================
# 9. Inference Example
# ============================
ner_pipe = pipeline(
    "ner",
    model=NER_MODEL,
    tokenizer=NER_MODEL,
    aggregation_strategy="simple"
)

sample_review = "ফেস ওয়াশ টা অনেক সুন্দর, অসংখ্য ধন্যবাদ ডেলিভারি খুব দ্রুত দেয়ার জন্য।"
print("\nSample review:", sample_review)
print("NER results (aggregated):", ner_pipe(sample_review))

sentiment_result = sentiment_pipe(sample_review, candidate_labels)
print(f"Sentiment: {sentiment_result['labels'][0]} (scores: {sentiment_result['scores']})")

print("\nAll zero‑shot tasks completed successfully!")


Training samples: 3000
Test samples:     30


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]


Computing NER predictions (zero‑shot)...

=== NER (binary O vs non‑O) Test Results ===
{'precision': 0.17549668874172186, 'recall': 1.0, 'f1': 0.29859154929577464, 'accuracy': 0.17549668874172186}

Computing sentiment predictions (zero‑shot)...

=== Sentiment Test Results ===
Accuracy: 0.6667
Weighted F1: 0.6479

Classification Report:
              precision    recall  f1-score   support

    negative       0.80      0.57      0.67         7
     neutral       0.00      0.00      0.00         5
    positive       0.76      0.89      0.82        18

    accuracy                           0.67        30
   macro avg       0.52      0.49      0.50        30
weighted avg       0.64      0.67      0.65        30


Predictions saved to zero_shot_predictions.csv


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Sample review: ফেস ওয়াশ টা অনেক সুন্দর, অসংখ্য ধন্যবাদ ডেলিভারি খুব দ্রুত দেয়ার জন্য।
NER results (aggregated): [{'entity_group': 'LABEL_5', 'score': np.float32(0.9401843), 'word': 'ফেস', 'start': 0, 'end': 3}, {'entity_group': 'LABEL_6', 'score': np.float32(0.9936067), 'word': 'ওযাশ টা', 'start': 4, 'end': 12}, {'entity_group': 'LABEL_0', 'score': np.float32(0.95081514), 'word': 'অনেক সনদর, অসংখয ধনযবাদ ডেলিভারি খব দরত দেযার জনয ।', 'start': 13, 'end': 72}]
Sentiment: positive (scores: [0.9146673679351807, 0.044046033173799515, 0.04128660261631012])

All zero‑shot tasks completed successfully!
